In [0]:
%python
data_source_uri = "s3://dalhussein-books/DEA-Book/datasets/school/v1/"
dataset_school = 'dbfs:/Workspace/Shared/DEA/datasets/school/'
checkpoint_path = 'dbfs:/Workspace/Shared/DEA/checkpoints/'
dlt_path = 'dbfs:/Workspace/Shared/DEA/dlt/'
db_name = 'DE_Associate_School'
dlt_db_name = 'school_dlt_db'
print(f"dataset.school = {dataset_school}")

In [0]:
%python
def clean_up():
    print("Removing Checkpoints ...")
    dbutils.fs.rm(checkpoint_path, True)
    print("Removing DLT storage location ...")
    dbutils.fs.rm(dlt_path, True)
    print("Dropping Database ...")
    spark.sql(f"DROP SCHEMA IF EXISTS {db_name} CASCADE")
    print("Dropping DLT database ...")
    spark.sql(f"DROP SCHEMA IF EXISTS {dlt_db_name} CASCADE")
    print("Removing Dataset ...")
    dbutils.fs.rm(dataset_school, True)
    print("Done")

In [0]:
%python
try:
    clean = int(dbutils.widgets.get("clean"))
except:
    clean = 0

if clean:
    clean_up()

In [0]:
%python
def path_exists(path):
  try:
    dbutils.fs.ls(path)
    return True
  except Exception as e:
    if 'java.io.FileNotFoundException' in str(e):
      return False
    else:
      raise

In [0]:
%python
def download_dataset(source, target):
    files = dbutils.fs.ls(source)

    for f in files:
        source_path = f"{source}/{f.name}"
        target_path = f"{target}/{f.name}"
        if not path_exists(target_path):
            print(f"Copying {f.name} ...")
            dbutils.fs.cp(source_path, target_path, True)

In [0]:
%python
def get_index(dir):
    files = dbutils.fs.ls(dir)
    index = 0
    if files:
        file = max(files).name
        index = int(file.rsplit('.', maxsplit=1)[0])
    return index+1

In [0]:
%python
# Structured Streaming
streaming_dir = f"{dataset_school}/enrollments-json-streaming"
raw_dir = f"{dataset_school}/enrollments-json-raw"

def load_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.json"
    print(f"Loading {latest_file} file to the school dataset")
    dbutils.fs.cp(f"{streaming_dir}/{latest_file}", f"{raw_dir}/{latest_file}")

    
def load_new_data(all=False):
    index = get_index(raw_dir)
    if index >= 10:
        print("No more data to load\n")

    elif all == True:
        while index <= 10:
            load_file(index)
            index += 1
    else:
        load_file(index)
        index += 1

In [0]:
%python
# DLT
streaming_enrollments_dir = f"{dataset_school}/enrollments-dlt-streaming"
streaming_courses_dir = f"{dataset_school}/courses-streaming"

raw_enrollments_dir = f"{dataset_school}/enrollments-dlt-raw"
raw_courses_dir = f"{dataset_school}/courses-cdc"

def load_json_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.json"
    print(f"Loading {latest_file} enrollments file to the school dataset")
    dbutils.fs.cp(f"{streaming_enrollments_dir}/{latest_file}", f"{raw_enrollments_dir}/{latest_file}")
    print(f"Loading {latest_file} courses file to the school dataset")
    dbutils.fs.cp(f"{streaming_courses_dir}/{latest_file}", f"{raw_courses_dir}/{latest_file}")

    
def load_new_json_data(all=False):
    index = get_index(raw_enrollments_dir)
    if index >= 10:
        print("No more data to load\n")

    elif all == True:
        while index <= 10:
            load_json_file(index)
            index += 1
    else:
        load_json_file(index)
        index += 1

In [0]:
%python
#clean_up()

In [0]:
%python
download_dataset(data_source_uri, dataset_school)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db_name}")
spark.sql(f"USE {db_name}")
print()
print(f"Schema name: workspace.{db_name}")

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_school}/students-json")
display(files)

In [0]:
SELECT * FROM workspace.default.students

In [0]:
USE workspace.default

In [0]:
DESCRIBE students

In [0]:
SELECT student_id, profile:first_name, profile:address:country
FROM students

In [0]:
SELECT profile 
FROM students 
LIMIT 1

In [0]:
CREATE OR REPLACE TEMP VIEW parsed_students AS
 SELECT student_id, from_json(profile, schema_of_json('{"first_name":"Sarah",
 "last_name":"Lundi", "gender":"Female", "address":{"street":"8 Greenbank Road",
 "city":"Ottawa", "country":"Canada"}}')) AS profile_struct
 FROM students;

SELECT * FROM parsed_students

In [0]:
DESCRIBE parsed_students

In [0]:
SELECT student_id, profile_struct.first_name, profile_struct.address.country
FROM parsed_students

In [0]:
CREATE OR REPLACE TEMP VIEW students_final AS
 SELECT student_id, profile_struct.*
 FROM parsed_students;

SELECT * FROM students_final

In [0]:
CREATE OR REPLACE TEMP VIEW students_final AS
 SELECT student_id, profile_struct.*
 FROM parsed_students;

SELECT * FROM students_final

In [0]:
SELECT enroll_id, student_id, courses
FROM enrollments

In [0]:
SELECT enroll_id, student_id, explode(courses) AS course
FROM enrollments

In [0]:
SELECT student_id,
 collect_set(enroll_id) AS enrollments_set,
 collect_set(courses.course_id) AS courses_set
FROM enrollments
GROUP BY student_id

In [0]:
SELECT student_id,
 collect_set(courses.course_id) As before_flatten,
 array_distinct(flatten(collect_set(courses.course_id))) AS after_flatten
FROM enrollments
GROUP BY student_id

In [0]:
CREATE OR REPLACE VIEW enrollments_enriched AS
SELECT *
FROM (
  SELECT *, explode(courses) AS course
  FROM enrollments) e
INNER JOIN courses_csv c
ON e.course.course_id = c.course_id;

SELECT * FROM enrollments_enriched

In [0]:
CREATE OR REPLACE TEMP VIEW enrollments_updates
AS SELECT * FROM parquet.`dbfs:/Workspace/Shared/DEA/datasets/school/enrollments-new`;

In [0]:
SELECT * FROM enrollments
UNION 
SELECT * FROM enrollments_updates

In [0]:
SELECT * FROM enrollments
UNION ALL
SELECT * FROM enrollments_updates

In [0]:
SELECT * FROM enrollments
INTERSECT
SELECT * FROM enrollments_updates

In [0]:
SELECT * FROM enrollments
MINUS
SELECT * FROM enrollments_updates

In [0]:
SELECT * FROM (
 SELECT student_id, course.course_id AS course_id, course.subtotal AS subtotal
 FROM enrollments_enriched
)
PIVOT (
 sum(subtotal) FOR course_id IN (
   'C01', 'C02', 'C03', 'C04', 'C05', 'C06',
   'C07', 'C08', 'C09', 'C10', 'C11', 'C12')
)